# Model Monitoring & Stability
## Production health of the loan-default PD model

This notebook turns the **`gold/monitoring`** and **`gold/predictions`** tables (produced every month by the Airflow pipeline) into the deck's monitoring story. It answers the three questions a model-risk committee asks about a deployed model:

1. **Is it still accurate?** — discrimination (ROC-AUC / Gini / KS) and calibration (Brier, actual vs predicted default rate) on loans whose labels have matured.
2. **Is the world still the same?** — population stability of the score and of key input features (PSI) vs the training baseline. This is available *immediately*, before labels mature, so it is our **early-warning** system.
3. **Is it still making money?** — approval rate and realised bad-rate among approved at the live decision threshold.

The bottom of the notebook formalises the **governance SOP**: PSI bands, retrain triggers, and escalation.

In [ ]:
import os
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

# Pick up edits to utils/* without requiring a full kernel restart.
for _name in list(sys.modules):
    if _name == "utils" or _name.startswith("utils."):
        del sys.modules[_name]

from sklearn.metrics import brier_score_loss, roc_auc_score

from utils.config import (
    GOLD_MONITORING_DIR, GOLD_PREDICTIONS_DIR,
    PSI_WARN_THRESHOLD, PSI_ALERT_THRESHOLD,
)
from utils.ml.dataset import load_training_frame, ks_statistic
from utils.ml.importance import load_monitored_features

sns.set_theme(style="white")
plt.rcParams.update(
    {
        "axes.grid": False,
        "axes.titlesize": 18,
        "axes.titleweight": "bold",
        "axes.labelsize": 15,
        "axes.labelweight": "bold",
        "xtick.labelsize": 13,
        "ytick.labelsize": 13,
        "legend.fontsize": 13,
    }
)

TITLE_FONTSIZE = 18
LABEL_FONTSIZE = 15
TICK_FONTSIZE = 13
LEGEND_FONTSIZE = 13


def style_axes(ax, *, title=None, xlabel=None, ylabel=None):
    ax.grid(False)
    if title is not None:
        ax.set_title(title, fontsize=TITLE_FONTSIZE, fontweight="bold", pad=12)
    if xlabel is not None:
        ax.set_xlabel(xlabel, fontsize=LABEL_FONTSIZE, fontweight="bold")
    if ylabel is not None:
        ax.set_ylabel(ylabel, fontsize=LABEL_FONTSIZE, fontweight="bold")
    ax.tick_params(axis="both", labelsize=TICK_FONTSIZE)
    legend = ax.get_legend()
    if legend is not None:
        for text in legend.get_texts():
            text.set_fontsize(LEGEND_FONTSIZE)


AS_OF = "2024-12-01"


def load_monitoring():
    files = sorted(GOLD_MONITORING_DIR.glob("snapshot_date=*/monitoring_metrics.parquet"))
    frames = [pd.read_parquet(f) for f in files]
    m = pd.concat(frames, ignore_index=True)
    m["snapshot_date"] = pd.to_datetime(m["snapshot_date"])
    return m.sort_values("snapshot_date")


def load_cohort_performance(as_of: str = AS_OF) -> pd.DataFrame:
    """Performance KPIs per loan *application* cohort (predictions.snapshot_date)."""
    pred_files = sorted(GOLD_PREDICTIONS_DIR.glob("snapshot_date=*/predictions.parquet"))
    preds = pd.concat([pd.read_parquet(f) for f in pred_files], ignore_index=True)
    preds["application_month"] = pd.to_datetime(preds["snapshot_date"])

    labels = load_training_frame(as_of)[["loan_id", "default_flag"]]
    scored = preds.merge(labels, on="loan_id", how="inner").dropna(subset=["default_flag"])

    rows = []
    for app_date, grp in scored.groupby("application_month"):
        if len(grp) < 10 or grp["default_flag"].nunique() < 2:
            continue
        y_true = grp["default_flag"].astype(int)
        y_score = grp["default_probability"]
        auc = float(roc_auc_score(y_true, y_score))
        thr = float(grp["decision_threshold"].iloc[0]) if "decision_threshold" in grp.columns else 0.5
        approved = (
            grp["default_prediction"] == 0
            if "default_prediction" in grp.columns
            else y_score < thr
        )
        rows.append(
            {
                "application_month": app_date,
                "roc_auc": auc,
                "gini": 2 * auc - 1,
                "ks": ks_statistic(y_true, y_score),
                "brier_score": brier_score_loss(y_true, y_score),
                "actual_default_rate": float(y_true.mean()),
                "predicted_default_rate": float((y_score >= 0.5).mean()),
                "approval_rate": float(approved.mean()),
                "bad_rate_among_approved": float(y_true[approved].mean()) if approved.any() else np.nan,
                "n_observations": len(grp),
            }
        )
    return pd.DataFrame(rows).set_index("application_month").sort_index()


mon = load_monitoring()
drift_wide = mon.pivot_table(index="snapshot_date", columns="metric_name", values="metric_value")
cohort = load_cohort_performance(AS_OF)
MONITORED_FEATURES = load_monitored_features()

print(
    "Application cohorts:",
    cohort.index.min().date(),
    "->",
    cohort.index.max().date(),
    f"({cohort.shape[0]} months with matured labels)",
)
print("Top features monitored for drift (champion importances):", MONITORED_FEATURES)
cohort[
    ["roc_auc", "gini", "ks", "brier_score", "actual_default_rate", "approval_rate"]
].dropna(how="all").round(3).tail(8)

## 1. Discrimination over time — is the model still accurate?

ROC-AUC, Gini and KS are computed **per loan application cohort** (`loan_start_date` / predictions `snapshot_date`) once that cohort's `30dpd@mob6` labels have matured. Flat lines = the model continues to rank risk as well as it did at training; a sustained decline is the trigger to investigate / retrain.

In [ ]:
perf = cohort[["roc_auc", "gini", "ks"]].dropna(how="all")
fig, ax = plt.subplots(figsize=(11, 5))
for col, c in zip(["roc_auc", "gini", "ks"], ["#4C72B0", "#55A868", "#C44E52"]):
    ax.plot(perf.index, perf[col], marker="o", label=col, color=c, linewidth=2.2, markersize=7)
style_axes(
    ax,
    title="Discrimination by application cohort",
    xlabel="loan application month",
    ylabel="metric value",
)
ax.set_ylim(0, 1)
ax.legend(frameon=False)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

print(perf.agg(["mean", "min", "max"]).round(3).T)

## 2. Calibration drift — actual vs predicted default rate

Discrimination (AUC) can stay high while the *level* of the PD drifts (e.g. macro conditions worsen). Tracking the realised default rate against the predicted rate detects this. A widening gap means recalibration is due, even if ranking is intact.

In [ ]:
cal = cohort[["actual_default_rate", "predicted_default_rate"]].dropna(how="all")
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(cal.index, cal["actual_default_rate"], marker="o", label="actual", linewidth=2.2, markersize=7)
ax.plot(cal.index, cal["predicted_default_rate"], marker="s", label="predicted", linewidth=2.2, markersize=7)
style_axes(
    ax,
    title="Actual vs predicted default rate by application cohort",
    xlabel="loan application month",
    ylabel="rate",
)
ax.legend(frameon=False)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 3. Stability / drift — PSI (the early-warning system)

Population Stability Index compares the *current* distribution to the *training-baseline* month. It needs **no labels**, so it flags problems months before performance metrics can. Convention: **<0.10 stable · 0.10–0.25 watch · >0.25 investigate/retrain**.

- **Score PSI** — has the model's output distribution shifted? (population mix change).
- **Feature PSI (CSI)** — PSI on the **top 7 champion-model inputs** (LightGBM split importance). In credit-risk practice this is often called **Characteristic Stability Index (CSI)**; the formula and thresholds are the same as PSI. The heatmap below shows these `feature_psi__*` metrics.

In [ ]:
# Score PSI with governance bands (indexed by application month).
if "score_psi" in drift_wide.columns:
    s = drift_wide["score_psi"].dropna()
    fig, ax = plt.subplots(figsize=(11, 5))
    ax.plot(s.index, s.values, marker="o", color="#4C72B0", linewidth=2.2, markersize=7)
    ax.axhline(
        PSI_WARN_THRESHOLD,
        ls="--",
        color="orange",
        linewidth=1.8,
        label=f"watch ({PSI_WARN_THRESHOLD})",
    )
    ax.axhline(
        PSI_ALERT_THRESHOLD,
        ls="--",
        color="red",
        linewidth=1.8,
        label=f"alert ({PSI_ALERT_THRESHOLD})",
    )
    style_axes(
        ax,
        title="Score PSI vs baseline month",
        xlabel="loan application month",
        ylabel="PSI",
    )
    ax.legend(frameon=False)
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()

# Feature PSI (CSI) on top-7 champion importances — application cohort vs baseline month.
feat = mon[mon["metric_name"].str.startswith("feature_psi__")].copy()
feat["feature"] = feat["metric_name"].str.replace("feature_psi__", "", regex=False)

latest_feat_month = feat["snapshot_date"].max()
feat_latest = (
    feat[feat["snapshot_date"] == latest_feat_month]
    .set_index("feature")
    .reindex(MONITORED_FEATURES)[["metric_value", "band"]]
    .rename(columns={"metric_value": "psi"})
    .round(3)
)
print(f"Top-7 feature CSI at latest snapshot ({latest_feat_month.date()}):")
display(feat_latest)

pivot = feat.pivot_table(
    index="feature",
    columns=feat["snapshot_date"].dt.strftime("%Y-%m"),
    values="metric_value",
)
if not pivot.empty:
    pivot = pivot.reindex([f for f in MONITORED_FEATURES if f in pivot.index])
    fig, ax = plt.subplots(figsize=(min(14, 2 + pivot.shape[1] * 0.9), 4 + pivot.shape[0] * 0.25))
    sns.heatmap(
        pivot,
        annot=True,
        fmt=".2f",
        cmap="rocket_r",
        vmin=0,
        vmax=0.5,
        cbar_kws={"label": "PSI / CSI"},
        ax=ax,
    )
    style_axes(
        ax,
        title="Top-7 champion feature drift (PSI = CSI)",
        xlabel="application month",
        ylabel="feature",
    )
    ax.tick_params(axis="x", labelsize=TICK_FONTSIZE, rotation=45)
    ax.tick_params(axis="y", labelsize=TICK_FONTSIZE)
    plt.tight_layout()
    plt.show()

## 4. Business value — approval rate, bad-rate among approved, and a gains table

AUC is abstract; the credit committee thinks in **approval rate** and **losses among approved**. At the live decision threshold, what share of applicants do we approve, and how risky is that approved book vs the through-the-door population? The **score-decile gains table** below is the classic scorecard artefact: it shows how concentrated the defaults are in the riskiest deciles (the basis for cut-off and limit strategy).

In [ ]:
kpi = cohort[["approval_rate", "bad_rate_among_approved"]].dropna(how="all")
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(kpi.index, kpi["approval_rate"], marker="o", label="approval rate", linewidth=2.2, markersize=7)
ax.plot(kpi.index, kpi["bad_rate_among_approved"], marker="s", label="bad-rate among approved", linewidth=2.2, markersize=7)
if "actual_default_rate" in cohort.columns:
    ax.plot(cohort.index, cohort["actual_default_rate"], ls="--", color="grey", label="portfolio bad rate", linewidth=2.0)
style_axes(
    ax,
    title="Business KPIs at the live decision threshold",
    xlabel="loan application month",
    ylabel="rate",
)
ax.legend(frameon=False)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

### Sealed OOT discrimination — reconcile with `model_metadata.json`

Section 1 reports ROC-AUC **per application month**. The sealed **OOT** window (most recent cohorts held out during training) is evaluated **pooled** below — the same definition as `train.py` / `02b`.

**Do not confuse with the gains table:** the `avg_pd` column there is the **mean predicted PD** in each score decile, not ROC-AUC. The deployed champion is the **calibrated** 36-feature LightGBM (`model_metadata.json`, OOT AUC ≈ 0.807). `02b` Section 11 reports the **uncalibrated** reduced model (OOT AUC ≈ 0.804) before isotonic calibration in `train.py`.


In [ ]:
from utils.config import MODEL_STORE_DIR, MODEL_METADATA_FILENAME, OOT_COHORTS
from utils.ml.dataset import random_dev_oot_split

_METADATA_PATH = MODEL_STORE_DIR / MODEL_METADATA_FILENAME
metadata = (
    json.loads(_METADATA_PATH.read_text(encoding="utf-8"))
    if _METADATA_PATH.exists()
    else {}
)

# OOT loan IDs — same temporal holdout as notebooks/02b and utils/ml/train.py
_, _, _, oot_df = random_dev_oot_split(load_training_frame(AS_OF), n_oot_cohorts=OOT_COHORTS)
oot_ids = set(oot_df["loan_id"])
oot_months = sorted(pd.to_datetime(oot_df["loan_start_date"]).dt.strftime("%Y-%m").unique())

pred_files = sorted(GOLD_PREDICTIONS_DIR.glob("snapshot_date=*/predictions.parquet"))
if not pred_files:
    print("No prediction partitions found under gold/predictions — run the pipeline backfill first.")
else:
    preds_all = pd.concat([pd.read_parquet(f) for f in pred_files], ignore_index=True)
    labels = load_training_frame(AS_OF)[["loan_id", "default_flag"]]
    scored_all = preds_all.merge(labels, on="loan_id", how="inner").dropna(subset=["default_flag"])
    oot_scored = scored_all[scored_all["loan_id"].isin(oot_ids)].copy()

    def _disc_metrics(y_true, y_score):
        y_true = y_true.astype(int)
        auc = float(roc_auc_score(y_true, y_score))
        return {
            "n": len(y_true),
            "roc_auc": auc,
            "gini": float(2 * auc - 1),
            "ks": float(ks_statistic(y_true, y_score)),
            "brier": float(brier_score_loss(y_true, y_score)),
        }

    rows = []
    if len(oot_scored) >= 10 and oot_scored["default_flag"].nunique() == 2:
        rows.append(
            {
                "source": "monitoring predictions (OOT pooled)",
                **_disc_metrics(oot_scored["default_flag"], oot_scored["default_probability"]),
            }
        )

    oot_meta = metadata.get("oot_metrics") or metadata.get("oot_test_metrics") or {}
    if oot_meta:
        rows.append(
            {
                "source": "model_metadata.json (deployed champion at train time)",
                "n": oot_meta.get("n"),
                "roc_auc": oot_meta.get("roc_auc"),
                "gini": oot_meta.get("gini"),
                "ks": oot_meta.get("ks"),
                "brier": oot_meta.get("brier"),
            }
        )

    rows.append(
        {
            "source": "02b Section 11 (uncalibrated reduced champion, notebook)",
            "n": 1502,
            "roc_auc": 0.8042,
            "gini": 0.6084,
            "ks": 0.4794,
            "brier": 0.1744,
        }
    )

    reconcile = pd.DataFrame(rows).set_index("source")
    print(f"Sealed OOT application months: {', '.join(oot_months)}")
    print(f"Scored OOT loans with matured labels: {len(oot_scored):,} / {len(oot_df):,} in split\n")

    if (
        "monitoring predictions (OOT pooled)" in reconcile.index
        and "model_metadata.json (deployed champion at train time)" in reconcile.index
    ):
        delta = (
            reconcile.loc["monitoring predictions (OOT pooled)", "roc_auc"]
            - reconcile.loc["model_metadata.json (deployed champion at train time)", "roc_auc"]
        )
        print(f"Monitoring vs metadata pooled OOT AUC delta: {delta:+.4f}\n")

    reconcile.round(4)


In [ ]:
# Score-decile gains table on all scored loans whose labels have matured.
pred_files = sorted(GOLD_PREDICTIONS_DIR.glob("snapshot_date=*/predictions.parquet"))
preds = pd.concat([pd.read_parquet(f) for f in pred_files], ignore_index=True)
preds = preds[preds["default_probability"].notna()]
labels = load_training_frame("2024-12-01")[["loan_id", "default_flag"]]
scored = preds.merge(labels, on="loan_id", how="inner").dropna(subset=["default_flag"])

scored["decile"] = pd.qcut(scored["default_probability"].rank(method="first"), 10,
                           labels=[f"D{i}" for i in range(10, 0, -1)])  # D1 = riskiest
gains = (scored.groupby("decile", observed=True)
               .agg(n=("default_flag", "size"),
                    defaults=("default_flag", "sum"),
                    bad_rate=("default_flag", "mean"),
                    avg_pd=("default_probability", "mean"))
               .sort_index())
total_bad = gains["defaults"].sum()
gains["pct_defaults_captured"] = (gains["defaults"].cumsum() / total_bad)
gains["lift"] = gains["bad_rate"] / scored["default_flag"].mean()
_pooled_auc = roc_auc_score(scored["default_flag"].astype(int), scored["default_probability"])
print(
    f"Scored loans with matured labels: {len(scored):,} | "
    f"pooled ROC-AUC={_pooled_auc:.3f} | "
    f"KS={ks_statistic(scored['default_flag'], scored['default_probability']):.3f}"
)
print("Note: avg_pd = mean predicted PD per decile (not ROC-AUC). Sealed OOT AUC: see reconciliation cell above.")
gains.round(3)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
gains["bad_rate"].plot(kind="bar", ax=ax[0], color="#C44E52")
ax[0].axhline(scored["default_flag"].mean(), ls="--", color="grey", label="avg", linewidth=2.0)
style_axes(
    ax[0],
    title="Bad rate by score decile (D1 = riskiest)",
    ylabel="bad rate",
)
ax[0].legend(frameon=False)
ax[0].tick_params(axis="x", labelsize=TICK_FONTSIZE)
cum = np.concatenate([[0], gains["pct_defaults_captured"].values])
ax[1].plot(np.linspace(0, 1, 11), cum, marker="o", label="model", linewidth=2.2, markersize=7)
ax[1].plot([0, 1], [0, 1], "--", color="grey", label="random", linewidth=2.0)
style_axes(
    ax[1],
    title="Cumulative gains curve",
    xlabel="share of population (riskiest first)",
    ylabel="share of defaults captured",
)
ax[1].legend(frameon=False)
plt.tight_layout()
plt.show()

## 5. Governance SOP — when do we act?

| Signal | Green (no action) | Amber (watch / investigate) | Red (act now) |
|---|---|---|---|
| **Score / feature PSI** | < 0.10 | 0.10 – 0.25 → root-cause review | > 0.25 → schedule retrain |
| **Gini / KS** | within ±5% of training | 5–10% relative drop → review | > 10% relative drop → retrain |
| **Actual vs predicted default rate** | gap < 2pp | 2–5pp → recalibrate | > 5pp → recalibrate + review |
| **Bad-rate among approved** | at/below plan | drifting up → tighten threshold | breach risk-appetite → escalate |

**Refresh cadence (implemented in the DAG).** Scheduled **quarterly retrain** (Jan/Apr/Jul/Oct) on the latest matured cohorts, plus **event-driven retrain** whenever a Red signal fires. Every candidate must beat the incumbent on OOT AUC **and** pass calibration before promotion (champion–challenger). All runs are versioned in the model store with metadata for audit.

The dashboard below is the one-line health check the pipeline writes to `outputs/charts/monitoring_summary.json` each month.

In [ ]:
latest = cohort.dropna(how="all").iloc[-1]

def band_psi(v):
    if pd.isna(v):
        return "n/a"
    return "RED" if v >= PSI_ALERT_THRESHOLD else ("AMBER" if v >= PSI_WARN_THRESHOLD else "GREEN")

psi_series = drift_wide["score_psi"].dropna() if "score_psi" in drift_wide.columns else pd.Series(dtype=float)
latest_psi = float(psi_series.iloc[-1]) if not psi_series.empty else np.nan
latest_psi_month = str(psi_series.index.max().date()) if not psi_series.empty else None

health = {
    "performance_as_of": str(cohort.index.max().date()),
    "score_psi_as_of": latest_psi_month,
    "gini": round(float(latest.get("gini", np.nan)), 3),
    "ks": round(float(latest.get("ks", np.nan)), 3),
    "brier": round(float(latest.get("brier_score", np.nan)), 3),
    "actual_default_rate": round(float(latest.get("actual_default_rate", np.nan)), 3),
    "approval_rate": round(float(latest.get("approval_rate", np.nan)), 3),
    "bad_rate_among_approved": round(float(latest.get("bad_rate_among_approved", np.nan)), 3),
    "score_psi": round(latest_psi, 3),
    "score_psi_band": band_psi(latest_psi),
}
print(json.dumps(health, indent=2))